# Real ODIR-5K federated trial on an A100
This notebook downloads the public ODIR-5K data, constructs patient-separated partitions, audits them, and executes the corrected federated baseline. It does not insert synthetic images or historical paper metrics. Select **Runtime → Change runtime type → A100 GPU** before running.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'

In [ ]:
!test -d /content/fundus-fl/.git || git clone --depth 1 https://github.com/ykumar2020/Enhanced-Privacy-Preserving-Architecture-for-Fundus-Disease-Diagnosis-with-Federated-Learning.git /content/fundus-fl
%cd /content/fundus-fl/reproduction
!python -m pip install -q -e ".[colab]"

## Acquire the real dataset
The public Kaggle distribution is downloaded into the temporary Colab runtime. If Kaggle asks for authentication, add a Colab secret named `KAGGLE_API_TOKEN` and rerun this cell. Never paste a token into the notebook.

In [ ]:
!retina-fl acquire-odir --destination /content/odir-data --template configs/corrected_paper_trial.yaml --config-out configs/runtime_colab_odir.yaml --run-root /content/odir-runs

In [ ]:
!retina-fl prepare --config configs/runtime_colab_odir.yaml
!retina-fl audit --manifest /content/odir-runs/manifest.csv

## Execute FedAvg
The corrected paper trial runs four clients, selects two clients per round, and uses five rounds with one local epoch, matching the reported training schedule while enforcing patient separation. Existing run evidence is not overwritten unless `--overwrite` is supplied.

In [ ]:
!retina-fl train federated --config configs/runtime_colab_odir.yaml

In [ ]:
import json
from pathlib import Path
run = Path('/content/odir-runs/federated')
metrics = json.loads((run / 'metrics.json').read_text())
communication = json.loads((run / 'communication.json').read_text())
print(json.dumps(metrics, indent=2))
print(json.dumps(communication, indent=2))
print('Evidence files:', *sorted(p.name for p in run.iterdir()), sep='\n- ')

## Optional matched baselines
Run these only when you want the corresponding real comparisons. They create separate evidence directories.
```bash
retina-fl train centralized --config configs/runtime_colab_odir.yaml
retina-fl train local --config configs/runtime_colab_odir.yaml
```